In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 22.1 - Overview, paths, and fixed UNMAPPED/MULTIMAPPED refinement design
# Purpose:
# Refine the two non-coordinate broad groups supported by Notebook 20:
# UNMAPPED and MULTIMAPPED.
#
# These unitigs cannot be subdivided reliably by one MG1655 coordinate.
# Instead, they are grouped phenotype-independently by carrier patterns
# across the same 176 blaTEM-1-only pathogens.
#
# For each parent group:
# 1. collapse exact duplicate 176-pathogen carrier patterns;
# 2. frequency-standardize each unique carrier pattern;
# 3. cluster unique carrier patterns into 10 groups;
# 4. weight unique patterns by original unitig multiplicity;
# 5. remove each cluster from the full Notebook 18 kernel;
# 6. compare the loss with matched random removals from the same parent.
#
# Random removals match exactly:
# - number of unitigs;
# - unitig presence-count distribution;
# - therefore kernel denominator contribution.
#
# BH correction is applied separately across 10 clusters within each parent.
# This notebook identifies carrier-pattern groups for further interpretation.
# It does not establish causal variants.

from pathlib import Path
import gc
import json
import time

import numpy as np
import pandas as pd
from scipy import sparse, optimize
from sklearn.cluster import MiniBatchKMeans
from IPython.display import display
PROJECT_ROOT = _repo_root()
NOTEBOOK_DIR = PROJECT_ROOT / "03_Notebooks" / "04_Genome_Comparison"
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"

NB18_DIR = PROJECT_ROOT / "04_Intermediate" / "18_Collective_Whole_Chromosome_Association"
NB18_K = NB18_DIR / "18_whole_sequence_unitig_similarity_matrix.npz"
NB18_SUMMARY = RESULTS_TABLE_DIR / "18_collective_unitig_association_summary.csv"
NB18_QC = RESULTS_TABLE_DIR / "18_collective_unitig_association_final_QC.csv"

NB19_DIR = PROJECT_ROOT / "04_Intermediate" / "19_Broad_Unitig_Ablation"
NB19_ASSIGNMENT = NB19_DIR / "19_unitig_reference_assignment.npz"
NB19_COMPONENTS = NB19_DIR / "19_group_kernel_components.npz"
NB19_GROUP_MANIFEST = RESULTS_TABLE_DIR / "19_broad_ablation_group_manifest.csv"
NB19_QC = RESULTS_TABLE_DIR / "19_broad_ablation_final_QC.csv"

NB20_FINAL_RESULTS = RESULTS_TABLE_DIR / "20_matched_random_ablation_final_results.csv"
NB20_QC = RESULTS_TABLE_DIR / "20_matched_random_ablation_final_QC.csv"

NB22_DIR = PROJECT_ROOT / "04_Intermediate" / "22_Unmapped_Multimapped_Carrier_Pattern_Refinement"
NB22_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_ASSIGNMENT = NB22_DIR / "22_carrier_pattern_cluster_assignment.npz"
CLUSTER_COMPONENTS = NB22_DIR / "22_carrier_pattern_cluster_kernel_components.npz"
PROGRESS_FILE = NB22_DIR / "22_matched_random_carrier_cluster_progress.csv.gz"
CLUSTER_MANIFEST = RESULTS_TABLE_DIR / "22_carrier_pattern_cluster_manifest.csv"
CLUSTER_SUMMARY = RESULTS_TABLE_DIR / "22_carrier_pattern_clustering_summary.csv"
OBSERVED_RESULTS = RESULTS_TABLE_DIR / "22_carrier_pattern_cluster_observed_results.csv"
SMOKE_TEST_FILE = RESULTS_TABLE_DIR / "22_carrier_pattern_cluster_smoke_test.csv"
FINAL_RESULTS = RESULTS_TABLE_DIR / "22_carrier_pattern_cluster_matched_random_results.csv"
FINAL_QC = RESULTS_TABLE_DIR / "22_carrier_pattern_cluster_final_QC.csv"
COMPLETION_FILE = NB22_DIR / "22_UNMAPPED_MULTIMAPPED_REFINEMENT_COMPLETE.json"

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
EXPECTED_BROAD_GROUPS = 12
EXPECTED_FULL_VARIANCE_FRACTION = 0.538815

PRIORITY_PARENTS = ["UNMAPPED", "MULTIMAPPED"]
N_CLUSTERS_PER_PARENT = 10
EXPECTED_CLUSTERS = len(PRIORITY_PARENTS) * N_CLUSTERS_PER_PARENT
N_RANDOM_PARTITIONS_PER_PARENT = 250

CLUSTER_RANDOM_SEED_BASE = 2026092200
BENCHMARK_RANDOM_SEED_BASE = 2026092250

for path in [
    PROJECT_ROOT, NOTEBOOK_DIR, RESULTS_TABLE_DIR,
    UNITIG_MATRIX, UNITIG_SAMPLES,
    NB18_K, NB18_SUMMARY, NB18_QC,
    NB19_ASSIGNMENT, NB19_COMPONENTS, NB19_GROUP_MANIFEST, NB19_QC,
    NB20_FINAL_RESULTS, NB20_QC,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 22 - UNMAPPED/MULTIMAPPED Carrier-Pattern Refinement")
print("Priority parent groups:", ", ".join(PRIORITY_PARENTS))
print("Carrier-pattern clusters per parent:", N_CLUSTERS_PER_PARENT)
print("Matched random partitions per parent:", N_RANDOM_PARTITIONS_PER_PARENT)
print("Clustering: frequency-standardized carrier patterns, weighted by unitig multiplicity")
print("BH correction: separately across 10 clusters within each parent")
print("\nTransition: Cell 22.2 will verify Notebooks 18-20 and load authoritative inputs.")


In [ ]:
#@title Cell 22.2 - Verify Notebooks 18-20 and load authoritative inputs
# Purpose:
# Confirm that UNMAPPED and MULTIMAPPED were supported by Notebook 20,
# recover the full Notebook 18 kernel decomposition, and load the complete
# unitig matrix and continuous MIC phenotype.

nb18_qc = pd.read_csv(NB18_QC)
nb19_qc = pd.read_csv(NB19_QC)
nb20_qc = pd.read_csv(NB20_QC)

assert len(nb18_qc) == 1 and bool(nb18_qc.loc[0, "final_QC_pass"])
assert len(nb19_qc) == 1 and bool(nb19_qc.loc[0, "final_QC_pass"])
assert len(nb20_qc) == 1 and bool(nb20_qc.loc[0, "final_QC_pass"])

nb18_summary = pd.read_csv(NB18_SUMMARY)
baseline_variance_fraction = float(
    nb18_summary.loc[0, "whole_sequence_unitig_variance_fraction"]
)
assert abs(baseline_variance_fraction - EXPECTED_FULL_VARIANCE_FRACTION) < 0.001

samples = pd.read_csv(UNITIG_SAMPLES).sort_values("sample_index").reset_index(drop=True)
assert len(samples) == EXPECTED_PATHOGENS
assert np.array_equal(samples["sample_index"].to_numpy(dtype=int), np.arange(EXPECTED_PATHOGENS))

y = samples["log2_mic"].to_numpy(dtype=float)
assert y.shape == (EXPECTED_PATHOGENS,)
assert np.isfinite(y).all()

X = sparse.load_npz(UNITIG_MATRIX).tocsc()
assert X.shape == (EXPECTED_PATHOGENS, EXPECTED_UNITIGS)

presence_count = np.asarray(X.sum(axis=0)).ravel().astype(np.int16)
assert presence_count.shape == (EXPECTED_UNITIGS,)
assert presence_count.min() >= 1
assert presence_count.max() <= EXPECTED_PATHOGENS - 1

with np.load(NB18_K) as archive:
    K_full = np.asarray(archive["K_unitig"], dtype=float)

K_full = (K_full + K_full.T) / 2.0
assert K_full.shape == (EXPECTED_PATHOGENS, EXPECTED_PATHOGENS)

group_manifest = (
    pd.read_csv(NB19_GROUP_MANIFEST)
    .sort_values("group_code")
    .reset_index(drop=True)
)
assert len(group_manifest) == EXPECTED_BROAD_GROUPS

with np.load(NB19_ASSIGNMENT) as archive:
    broad_group_code = np.asarray(archive["group_code"], dtype=np.uint8)

assert broad_group_code.shape == (EXPECTED_UNITIGS,)

with np.load(NB19_COMPONENTS) as archive:
    broad_group_numerators = np.asarray(archive["group_numerators"], dtype=float)
    broad_group_denominators = np.asarray(archive["group_denominators"], dtype=float)
    broad_group_unitig_counts = np.asarray(archive["group_unitig_counts"], dtype=np.int64)

assert broad_group_numerators.shape == (
    EXPECTED_BROAD_GROUPS,
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

total_numerator = np.sum(broad_group_numerators, axis=0)
total_denominator = float(np.sum(broad_group_denominators))

K_reconstructed = total_numerator / total_denominator
K_reconstructed = (K_reconstructed + K_reconstructed.T) / 2.0

full_reconstruction_error = float(np.max(np.abs(K_reconstructed - K_full)))
assert full_reconstruction_error < 1e-10

nb20_results = pd.read_csv(NB20_FINAL_RESULTS)

priority_check = nb20_results.loc[
    nb20_results["group_name"].isin(PRIORITY_PARENTS),
    [
        "group_name",
        "observed_drop_after_removal",
        "empirical_p_value",
        "Benjamini_Hochberg_q_value",
        "larger_drop_than_matched_random_after_BH",
    ],
].copy()

assert set(priority_check["group_name"]) == set(PRIORITY_PARENTS)
assert priority_check["larger_drop_than_matched_random_after_BH"].astype(bool).all()

parent_code_by_name = {
    str(row["group_name"]): int(row["group_code"])
    for _, row in group_manifest.iterrows()
}

for parent_name in PRIORITY_PARENTS:
    assert parent_name in parent_code_by_name

print("Notebook 18 QC: PASS")
print("Notebook 19 QC: PASS")
print("Notebook 20 QC: PASS")
print("Full variance fraction:", baseline_variance_fraction)
print("Full K reconstruction error:", full_reconstruction_error)
print("\nNon-coordinate parent groups confirmed for refinement:")
display(priority_check.sort_values("group_name").reset_index(drop=True))

print("\nCell 22.2 complete.")
print("Transition: Cell 22.3 will cluster UNMAPPED and MULTIMAPPED by carrier-pattern structure.")


In [ ]:
#@title Cell 22.3 - Cluster UNMAPPED and MULTIMAPPED by carrier-pattern structure
# Purpose:
# Partition each parent into 10 phenotype-independent groups.
#
# Exact duplicate carrier patterns are collapsed first.
# For each unique carrier vector x:
#   p = mean(x)
#   z = (x - p) / sqrt[p(1-p)]
#
# Every standardized binary pattern has the same Euclidean norm, so
# clustering is driven by which pathogens share the pattern rather than
# simply by raw frequency.
#
# Unique patterns are weighted by their original unitig multiplicity.
# Cluster numbering is then ordered by decreasing unitig count.
# MIC is not used in this cell.

cluster_assignment = np.full(EXPECTED_UNITIGS, -1, dtype=np.int16)

manifest_rows = []
summary_rows = []
global_cluster_code = 0
clustering_start = time.time()

for parent_order, parent_name in enumerate(PRIORITY_PARENTS):
    parent_code = parent_code_by_name[parent_name]
    parent_indices = np.flatnonzero(broad_group_code == parent_code)

    assert len(parent_indices) == int(broad_group_unitig_counts[parent_code])

    print("\nClustering", parent_name, "- unitigs:", f"{len(parent_indices):,}")

    parent_binary = (
        X[:, parent_indices]
        .T
        .toarray()
        .astype(np.uint8, copy=False)
    )

    assert parent_binary.shape == (len(parent_indices), EXPECTED_PATHOGENS)

    packed = np.packbits(parent_binary, axis=1, bitorder="little")

    _, unique_first_index, unique_inverse, unique_multiplicity = np.unique(
        packed,
        axis=0,
        return_index=True,
        return_inverse=True,
        return_counts=True,
    )

    del packed
    gc.collect()

    unique_binary = parent_binary[unique_first_index].astype(np.float32, copy=False)
    del parent_binary
    gc.collect()

    n_unique_patterns = unique_binary.shape[0]
    assert n_unique_patterns >= N_CLUSTERS_PER_PARENT

    unique_presence_count = unique_binary.sum(axis=1).astype(np.float32)
    unique_p = unique_presence_count / EXPECTED_PATHOGENS

    assert np.all(unique_p > 0)
    assert np.all(unique_p < 1)

    scale = np.sqrt(unique_p * (1.0 - unique_p)).astype(np.float32)

    standardized_patterns = (
        unique_binary - unique_p[:, None]
    ) / scale[:, None]

    pattern_norms = np.linalg.norm(standardized_patterns, axis=1)
    expected_norm = np.sqrt(EXPECTED_PATHOGENS)
    maximum_norm_deviation = float(
        np.max(np.abs(pattern_norms - expected_norm))
    )
    assert maximum_norm_deviation < 1e-3

    model = MiniBatchKMeans(
        n_clusters=N_CLUSTERS_PER_PARENT,
        random_state=CLUSTER_RANDOM_SEED_BASE + parent_order,
        batch_size=8192,
        n_init=10,
        max_iter=200,
        reassignment_ratio=0.01,
    )

    model.fit(
        standardized_patterns,
        sample_weight=unique_multiplicity.astype(np.float64),
    )

    raw_unique_labels = model.labels_.astype(np.int16)
    raw_unitig_labels = raw_unique_labels[unique_inverse]

    raw_unitig_counts = np.bincount(
        raw_unitig_labels,
        minlength=N_CLUSTERS_PER_PARENT,
    ).astype(np.int64)

    assert np.all(raw_unitig_counts > 0)

    raw_cluster_order = np.lexsort(
        (
            np.arange(N_CLUSTERS_PER_PARENT),
            -raw_unitig_counts,
        )
    )

    raw_to_local = np.empty(N_CLUSTERS_PER_PARENT, dtype=np.int16)
    raw_to_local[raw_cluster_order] = np.arange(
        N_CLUSTERS_PER_PARENT,
        dtype=np.int16,
    )

    unique_local_labels = raw_to_local[raw_unique_labels]
    unitig_local_labels = raw_to_local[raw_unitig_labels]

    parent_global_codes = np.arange(
        global_cluster_code,
        global_cluster_code + N_CLUSTERS_PER_PARENT,
        dtype=np.int16,
    )

    cluster_assignment[parent_indices] = parent_global_codes[unitig_local_labels]

    for local_cluster_code in range(N_CLUSTERS_PER_PARENT):
        current_global_code = int(parent_global_codes[local_cluster_code])

        unitig_mask = unitig_local_labels == local_cluster_code
        unique_mask = unique_local_labels == local_cluster_code

        current_unitig_indices = parent_indices[unitig_mask]
        current_presence_counts = presence_count[current_unitig_indices]

        assert len(current_unitig_indices) > 0

        p_current = current_presence_counts.astype(np.float64) / EXPECTED_PATHOGENS
        denominator_current = float(np.sum(p_current * (1.0 - p_current)))

        cluster_name = f"{parent_name}_C{local_cluster_code + 1:02d}"

        manifest_rows.append(
            {
                "global_cluster_code": current_global_code,
                "parent_order": parent_order,
                "parent_group_code": parent_code,
                "parent_group_name": parent_name,
                "local_cluster_code": local_cluster_code,
                "cluster_name": cluster_name,
                "n_unitigs": len(current_unitig_indices),
                "n_unique_carrier_patterns": int(np.sum(unique_mask)),
                "fraction_of_parent_unitigs": (
                    len(current_unitig_indices) / len(parent_indices)
                ),
                "kernel_denominator_contribution": denominator_current,
                "minimum_presence_count": int(current_presence_counts.min()),
                "maximum_presence_count": int(current_presence_counts.max()),
            }
        )

    summary_rows.append(
        {
            "parent_group_name": parent_name,
            "n_unitigs": len(parent_indices),
            "n_unique_carrier_patterns": n_unique_patterns,
            "exact_duplicate_unitigs": len(parent_indices) - n_unique_patterns,
            "clusters": N_CLUSTERS_PER_PARENT,
            "maximum_standardized_pattern_norm_deviation": maximum_norm_deviation,
            "MiniBatchKMeans_inertia": float(model.inertia_),
        }
    )

    print("Unique carrier patterns:", f"{n_unique_patterns:,}")
    print("Exact duplicate unitigs:", f"{len(parent_indices) - n_unique_patterns:,}")
    print("Carrier-pattern clustering: PASS")

    global_cluster_code += N_CLUSTERS_PER_PARENT

    del unique_binary
    del standardized_patterns
    del unique_inverse
    del unique_multiplicity
    del raw_unique_labels
    del raw_unitig_labels
    del unique_local_labels
    del unitig_local_labels
    del model
    gc.collect()

assert global_cluster_code == EXPECTED_CLUSTERS

selected_parent_codes = [
    parent_code_by_name[name]
    for name in PRIORITY_PARENTS
]

selected_parent_mask = np.isin(
    broad_group_code,
    selected_parent_codes,
)

assert np.all(cluster_assignment[selected_parent_mask] >= 0)
assert np.all(cluster_assignment[~selected_parent_mask] == -1)

cluster_manifest = pd.DataFrame(manifest_rows)
clustering_summary = pd.DataFrame(summary_rows)

assert len(cluster_manifest) == EXPECTED_CLUSTERS

for parent_name in PRIORITY_PARENTS:
    parent_code = parent_code_by_name[parent_name]
    manifest_count = int(
        cluster_manifest.loc[
            cluster_manifest["parent_group_name"] == parent_name,
            "n_unitigs",
        ].sum()
    )
    assert manifest_count == int(broad_group_unitig_counts[parent_code])

np.savez_compressed(
    CLUSTER_ASSIGNMENT,
    cluster_assignment=cluster_assignment,
    priority_parent_names=np.asarray(PRIORITY_PARENTS, dtype="U16"),
)

cluster_manifest.to_csv(CLUSTER_MANIFEST, index=False)
clustering_summary.to_csv(CLUSTER_SUMMARY, index=False)

print("\nCarrier-pattern clustering summary:")
display(clustering_summary)

print("\nCluster manifest:")
display(cluster_manifest)

print(
    "\nClustering elapsed:",
    f"{(time.time() - clustering_start) / 60.0:.1f} minutes",
)

print("\nCell 22.3 complete.")
print(
    "Transition: Cell 22.4 will decompose the 20 clusters, verify exact "
    "parent-kernel reconstruction, and fit observed leave-one-cluster-out effects."
)


In [ ]:
#@title Cell 22.4 - Decompose carrier-pattern cluster kernels and fit observed ablations
# Purpose:
# Calculate the exact kernel contribution of every carrier-pattern cluster.
#
# The 10 clusters within each parent must exactly reconstruct that parent's
# Notebook 19 kernel component.
#
# Then remove each cluster from the full Notebook 18 kernel and measure the
# observed fall in REML variance fraction.

def numerator_from_columns(columns):
    X_group = X[:, columns]

    count_group = presence_count[columns].astype(np.float64)
    p_group = count_group / EXPECTED_PATHOGENS

    denominator_group = float(
        np.sum(p_group * (1.0 - p_group))
    )

    X_int = X_group.astype(np.int32)
    XX_group = (X_int @ X_int.T).toarray().astype(np.float64)

    Xp_group = np.asarray(
        X_group @ p_group
    ).reshape(-1).astype(np.float64)

    p2_group = float(p_group @ p_group)

    numerator_group = (
        XX_group
        - Xp_group[:, None]
        - Xp_group[None, :]
        + p2_group
    )

    numerator_group = (numerator_group + numerator_group.T) / 2.0

    return numerator_group, denominator_group

def prepare_kernel(K):
    K = np.asarray(K, dtype=float)
    K = (K + K.T) / 2.0

    eigenvalues, eigenvectors = np.linalg.eigh(K)

    minimum_eigenvalue = float(eigenvalues.min())
    if minimum_eigenvalue < -1e-6:
        raise ValueError(
            "Kernel is not positive semidefinite: "
            f"minimum eigenvalue = {minimum_eigenvalue}"
        )

    eigenvalues = np.maximum(eigenvalues, 0.0)

    transformed_intercept = (
        eigenvectors.T
        @ np.ones(K.shape[0], dtype=float)
    )

    return {
        "eigenvalues": eigenvalues,
        "eigenvectors": eigenvectors,
        "transformed_intercept": transformed_intercept,
    }

def fit_null_reml_prepared(y_input, prepared):
    y_input = np.asarray(y_input, dtype=float).reshape(-1)

    eigenvalues = prepared["eigenvalues"]
    eigenvectors = prepared["eigenvectors"]
    transformed_intercept = prepared["transformed_intercept"]

    transformed_y = eigenvectors.T @ y_input

    n = len(y_input)
    degrees_of_freedom = n - 1

    def evaluate_ratio(ratio):
        if ratio < 0:
            return None

        covariance_eigenvalues = 1.0 + ratio * eigenvalues

        if np.any(covariance_eigenvalues <= 0):
            return None

        inverse_weights = 1.0 / covariance_eigenvalues

        information = float(
            np.sum(
                transformed_intercept
                * transformed_intercept
                * inverse_weights
            )
        )

        if information <= 0:
            return None

        beta_0 = float(
            np.sum(
                transformed_intercept
                * transformed_y
                * inverse_weights
            )
            / information
        )

        transformed_residual = (
            transformed_y
            - beta_0 * transformed_intercept
        )

        residual_quadratic = float(
            np.sum(
                transformed_residual
                * transformed_residual
                * inverse_weights
            )
        )

        if residual_quadratic <= 0:
            return None

        sigma_e2 = residual_quadratic / degrees_of_freedom
        sigma_g2 = ratio * sigma_e2

        objective = 0.5 * (
            degrees_of_freedom * np.log(sigma_e2)
            + np.log(covariance_eigenvalues).sum()
            + np.log(information)
        )

        return {
            "objective": float(objective),
            "variance_fraction": float(
                sigma_g2 / (sigma_g2 + sigma_e2)
            ),
        }

    def objective_on_log_ratio(log_ratio):
        result = evaluate_ratio(np.exp(log_ratio))
        if result is None:
            return np.inf
        return result["objective"]

    optimized = optimize.minimize_scalar(
        objective_on_log_ratio,
        bounds=(-12.0, 12.0),
        method="bounded",
        options={"xatol": 1e-8, "maxiter": 500},
    )

    candidates = []

    zero_result = evaluate_ratio(0.0)
    if zero_result is not None:
        candidates.append(zero_result)

    if optimized.success:
        optimized_result = evaluate_ratio(float(np.exp(optimized.x)))
        if optimized_result is not None:
            candidates.append(optimized_result)

    high_result = evaluate_ratio(float(np.exp(12.0)))
    if high_result is not None:
        candidates.append(high_result)

    assert candidates

    return min(candidates, key=lambda item: item["objective"])

cluster_numerators = np.zeros(
    (
        EXPECTED_CLUSTERS,
        EXPECTED_PATHOGENS,
        EXPECTED_PATHOGENS,
    ),
    dtype=np.float64,
)

cluster_denominators = np.zeros(
    EXPECTED_CLUSTERS,
    dtype=np.float64,
)

decomposition_start = time.time()

for cluster_index in range(EXPECTED_CLUSTERS):
    columns = np.flatnonzero(cluster_assignment == cluster_index)
    assert len(columns) > 0

    numerator_group, denominator_group = numerator_from_columns(columns)

    cluster_numerators[cluster_index] = numerator_group
    cluster_denominators[cluster_index] = denominator_group

parent_reconstruction_rows = []

for parent_name in PRIORITY_PARENTS:
    parent_code = parent_code_by_name[parent_name]

    cluster_codes = (
        cluster_manifest.loc[
            cluster_manifest["parent_group_name"] == parent_name,
            "global_cluster_code",
        ]
        .to_numpy(dtype=int)
    )

    reconstructed_parent_numerator = np.sum(
        cluster_numerators[cluster_codes],
        axis=0,
    )

    reconstructed_parent_denominator = float(
        np.sum(cluster_denominators[cluster_codes])
    )

    numerator_error = float(
        np.max(
            np.abs(
                reconstructed_parent_numerator
                - broad_group_numerators[parent_code]
            )
        )
    )

    denominator_error = float(
        abs(
            reconstructed_parent_denominator
            - broad_group_denominators[parent_code]
        )
    )

    assert numerator_error < 1e-8
    assert denominator_error < 1e-8

    parent_reconstruction_rows.append(
        {
            "parent_group_name": parent_name,
            "maximum_numerator_reconstruction_error": numerator_error,
            "denominator_reconstruction_error": denominator_error,
        }
    )

parent_reconstruction = pd.DataFrame(parent_reconstruction_rows)

cluster_manifest = cluster_manifest.copy()
cluster_manifest["kernel_denominator_contribution"] = cluster_denominators

for parent_name in PRIORITY_PARENTS:
    parent_code = parent_code_by_name[parent_name]
    mask = cluster_manifest["parent_group_name"] == parent_name

    cluster_manifest.loc[
        mask,
        "fraction_of_parent_kernel_denominator",
    ] = (
        cluster_manifest.loc[
            mask,
            "kernel_denominator_contribution",
        ]
        / broad_group_denominators[parent_code]
    )

cluster_manifest.to_csv(CLUSTER_MANIFEST, index=False)

np.savez_compressed(
    CLUSTER_COMPONENTS,
    cluster_numerators=cluster_numerators,
    cluster_denominators=cluster_denominators,
)

observed_rows = []

for cluster_index in range(EXPECTED_CLUSTERS):
    denominator_group = float(cluster_denominators[cluster_index])

    leave_out_K = (
        total_numerator
        - cluster_numerators[cluster_index]
    ) / (
        total_denominator
        - denominator_group
    )

    leave_out_K = (leave_out_K + leave_out_K.T) / 2.0

    prepared = prepare_kernel(leave_out_K)
    fit = fit_null_reml_prepared(y, prepared)

    leave_out_fraction = float(fit["variance_fraction"])
    observed_drop = baseline_variance_fraction - leave_out_fraction

    manifest_row = cluster_manifest.loc[
        cluster_manifest["global_cluster_code"] == cluster_index
    ].iloc[0]

    observed_rows.append(
        {
            "global_cluster_code": cluster_index,
            "parent_group_name": str(manifest_row["parent_group_name"]),
            "local_cluster_code": int(manifest_row["local_cluster_code"]),
            "cluster_name": str(manifest_row["cluster_name"]),
            "n_unitigs": int(manifest_row["n_unitigs"]),
            "n_unique_carrier_patterns": int(
                manifest_row["n_unique_carrier_patterns"]
            ),
            "fraction_of_parent_kernel_denominator": float(
                manifest_row["fraction_of_parent_kernel_denominator"]
            ),
            "leave_one_cluster_out_variance_fraction": leave_out_fraction,
            "absolute_drop_after_removal": observed_drop,
        }
    )

observed_results = pd.DataFrame(observed_rows)
observed_results.to_csv(OBSERVED_RESULTS, index=False)

print("Parent-kernel reconstruction: PASS")
display(parent_reconstruction)

print("\nObserved carrier-pattern cluster ablations, ranked within each parent:")
display(
    observed_results
    .sort_values(
        ["parent_group_name", "absolute_drop_after_removal"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print(
    "\nDecomposition and observed fitting elapsed:",
    f"{(time.time() - decomposition_start) / 60.0:.1f} minutes",
)

print("\nCell 22.4 complete.")
print(
    "Transition: Cell 22.5 will run one matched-random partition "
    "inside each parent as a smoke test."
)


In [ ]:
#@title Cell 22.5 - Matched-random smoke test within each non-coordinate parent
# Purpose:
# Verify the parent-specific matched-random design before the long benchmark.
#
# Random cluster labels are shuffled only among unitigs with the same
# presence count across the 176 pathogens.
#
# Every random counterpart therefore preserves exactly:
# - observed number of unitigs;
# - observed presence-count histogram;
# - observed kernel denominator contribution.

parent_randomization_data = {}

for parent_order, parent_name in enumerate(PRIORITY_PARENTS):
    parent_code = parent_code_by_name[parent_name]
    parent_unitig_indices = np.flatnonzero(
        broad_group_code == parent_code
    )

    parent_cluster_rows = (
        cluster_manifest.loc[
            cluster_manifest["parent_group_name"] == parent_name
        ]
        .sort_values("local_cluster_code")
        .reset_index(drop=True)
    )

    assert len(parent_cluster_rows) == N_CLUSTERS_PER_PARENT

    global_codes = parent_cluster_rows[
        "global_cluster_code"
    ].to_numpy(dtype=int)

    global_to_local = {
        int(global_code): int(local_code)
        for global_code, local_code in zip(
            global_codes,
            parent_cluster_rows[
                "local_cluster_code"
            ].to_numpy(dtype=int),
        )
    }

    observed_local_code = np.asarray(
        [
            global_to_local[int(global_code)]
            for global_code in cluster_assignment[
                parent_unitig_indices
            ]
        ],
        dtype=np.int8,
    )

    parent_presence_count = presence_count[
        parent_unitig_indices
    ]

    stratum_positions = {}
    stratum_local_counts = {}

    for count in np.unique(parent_presence_count):
        positions = np.flatnonzero(
            parent_presence_count == count
        )

        local_counts = np.bincount(
            observed_local_code[positions],
            minlength=N_CLUSTERS_PER_PARENT,
        ).astype(np.int64)

        assert int(local_counts.sum()) == len(positions)

        stratum_positions[int(count)] = positions
        stratum_local_counts[int(count)] = local_counts

    reconstructed_counts = np.zeros(
        N_CLUSTERS_PER_PARENT,
        dtype=np.int64,
    )

    reconstructed_denominators = np.zeros(
        N_CLUSTERS_PER_PARENT,
        dtype=np.float64,
    )

    for count, local_counts in stratum_local_counts.items():
        p = count / EXPECTED_PATHOGENS
        weight = p * (1.0 - p)

        reconstructed_counts += local_counts
        reconstructed_denominators += local_counts * weight

    observed_counts = parent_cluster_rows[
        "n_unitigs"
    ].to_numpy(dtype=np.int64)

    observed_denominators = cluster_denominators[
        global_codes
    ]

    assert np.array_equal(
        reconstructed_counts,
        observed_counts,
    )

    assert np.allclose(
        reconstructed_denominators,
        observed_denominators,
        atol=1e-10,
        rtol=1e-12,
    )

    parent_randomization_data[parent_name] = {
        "parent_order": parent_order,
        "parent_unitig_indices": parent_unitig_indices,
        "global_codes": global_codes,
        "observed_counts": observed_counts,
        "observed_denominators": observed_denominators,
        "stratum_positions": stratum_positions,
        "stratum_local_counts": stratum_local_counts,
    }

def matched_random_local_code(parent_name, replicate_index):
    info = parent_randomization_data[parent_name]

    rng = np.random.default_rng(
        BENCHMARK_RANDOM_SEED_BASE
        + int(info["parent_order"]) * 100_000
        + int(replicate_index)
    )

    random_local_code = np.empty(
        len(info["parent_unitig_indices"]),
        dtype=np.int8,
    )

    for count, positions in info["stratum_positions"].items():
        local_counts = info["stratum_local_counts"][count]

        labels = np.repeat(
            np.arange(
                N_CLUSTERS_PER_PARENT,
                dtype=np.int8,
            ),
            local_counts,
        )

        assert len(labels) == len(positions)

        rng.shuffle(labels)
        random_local_code[positions] = labels

    return random_local_code

smoke_start = time.time()
smoke_rows = []

for parent_name in PRIORITY_PARENTS:
    info = parent_randomization_data[parent_name]

    random_local_code = matched_random_local_code(
        parent_name,
        0,
    )

    random_counts = np.bincount(
        random_local_code,
        minlength=N_CLUSTERS_PER_PARENT,
    ).astype(np.int64)

    assert np.array_equal(
        random_counts,
        info["observed_counts"],
    )

    for local_code in range(N_CLUSTERS_PER_PARENT):
        columns = info[
            "parent_unitig_indices"
        ][
            random_local_code == local_code
        ]

        random_numerator, random_denominator = numerator_from_columns(
            columns
        )

        expected_denominator = float(
            info["observed_denominators"][local_code]
        )

        assert abs(
            random_denominator
            - expected_denominator
        ) < 1e-10

        leave_out_K = (
            total_numerator
            - random_numerator
        ) / (
            total_denominator
            - expected_denominator
        )

        leave_out_K = (leave_out_K + leave_out_K.T) / 2.0

        prepared = prepare_kernel(leave_out_K)
        fit = fit_null_reml_prepared(y, prepared)

        random_fraction = float(
            fit["variance_fraction"]
        )

        random_drop = (
            baseline_variance_fraction
            - random_fraction
        )

        global_code = int(
            info["global_codes"][local_code]
        )

        cluster_name = str(
            cluster_manifest.loc[
                cluster_manifest["global_cluster_code"]
                == global_code,
                "cluster_name",
            ].iloc[0]
        )

        smoke_rows.append(
            {
                "parent_group_name": parent_name,
                "local_cluster_code": local_code,
                "cluster_name": cluster_name,
                "matched_unitigs": len(columns),
                "random_leave_one_cluster_out_variance_fraction": random_fraction,
                "random_drop_from_full": random_drop,
            }
        )

smoke_results = pd.DataFrame(smoke_rows)

smoke_elapsed_seconds = time.time() - smoke_start
estimated_total_minutes = (
    smoke_elapsed_seconds
    * N_RANDOM_PARTITIONS_PER_PARENT
    / 60.0
)

smoke_results.to_csv(
    SMOKE_TEST_FILE,
    index=False,
)

print("Matched-random carrier-cluster smoke test: PASS")
print(
    "One partition for both parents elapsed:",
    f"{smoke_elapsed_seconds:.1f}",
    "seconds",
)
print(
    "Estimated 250-partition total runtime:",
    f"{estimated_total_minutes:.1f}",
    "minutes",
)
print(
    "The long benchmark is restartable after every completed parent-partition."
)

display(smoke_results)

print("\nCell 22.5 complete.")
print(
    "Transition: Cell 22.6 will run or resume 250 matched-random "
    "partitions within each non-coordinate parent."
)


In [ ]:
#@title Cell 22.6 - Run or resume matched-random carrier-cluster benchmarks
# Purpose:
# Generate 250 matched-random partitions inside each of the two parent groups.
#
# Progress is saved after every completed parent-partition.
# If Colab disconnects, rerun Cells 22.1-22.6; completed work is skipped.

progress_columns = [
    "parent_group_name",
    "replicate",
    "local_cluster_code",
    "cluster_name",
    "random_leave_one_cluster_out_variance_fraction",
    "random_drop_from_full",
]

if PROGRESS_FILE.exists():
    progress = pd.read_csv(PROGRESS_FILE)
else:
    progress = pd.DataFrame(columns=progress_columns)

completed_parent_replicates = set()

if len(progress) > 0:
    completed_counts = (
        progress.groupby(
            ["parent_group_name", "replicate"]
        )[
            "local_cluster_code"
        ]
        .nunique()
    )

    completed_parent_replicates = set(
        (
            str(parent_name),
            int(replicate),
        )
        for (
            parent_name,
            replicate
        ), count in completed_counts.items()
        if count == N_CLUSTERS_PER_PARENT
    )

total_parent_partitions = (
    len(PRIORITY_PARENTS)
    * N_RANDOM_PARTITIONS_PER_PARENT
)

print(
    "Already completed parent-partitions:",
    len(completed_parent_replicates),
    "/",
    total_parent_partitions,
)

benchmark_start = time.time()

for parent_name in PRIORITY_PARENTS:
    info = parent_randomization_data[parent_name]

    for replicate_index in range(
        N_RANDOM_PARTITIONS_PER_PARENT
    ):
        key = (
            parent_name,
            replicate_index,
        )

        if key in completed_parent_replicates:
            continue

        replicate_start = time.time()

        random_local_code = matched_random_local_code(
            parent_name,
            replicate_index,
        )

        random_counts = np.bincount(
            random_local_code,
            minlength=N_CLUSTERS_PER_PARENT,
        ).astype(np.int64)

        assert np.array_equal(
            random_counts,
            info["observed_counts"],
        )

        replicate_rows = []

        for local_code in range(
            N_CLUSTERS_PER_PARENT
        ):
            columns = info[
                "parent_unitig_indices"
            ][
                random_local_code == local_code
            ]

            random_numerator, random_denominator = numerator_from_columns(
                columns
            )

            expected_denominator = float(
                info["observed_denominators"][local_code]
            )

            assert abs(
                random_denominator
                - expected_denominator
            ) < 1e-10

            leave_out_K = (
                total_numerator
                - random_numerator
            ) / (
                total_denominator
                - expected_denominator
            )

            leave_out_K = (
                leave_out_K
                + leave_out_K.T
            ) / 2.0

            prepared = prepare_kernel(
                leave_out_K
            )

            fit = fit_null_reml_prepared(
                y,
                prepared,
            )

            random_fraction = float(
                fit["variance_fraction"]
            )

            random_drop = (
                baseline_variance_fraction
                - random_fraction
            )

            global_code = int(
                info["global_codes"][local_code]
            )

            cluster_name = str(
                cluster_manifest.loc[
                    cluster_manifest[
                        "global_cluster_code"
                    ]
                    == global_code,
                    "cluster_name",
                ].iloc[0]
            )

            replicate_rows.append(
                {
                    "parent_group_name": parent_name,
                    "replicate": replicate_index,
                    "local_cluster_code": local_code,
                    "cluster_name": cluster_name,
                    "random_leave_one_cluster_out_variance_fraction": random_fraction,
                    "random_drop_from_full": random_drop,
                }
            )

        if len(progress) > 0:
            keep_mask = ~(
                (
                    progress[
                        "parent_group_name"
                    ].astype(str)
                    == parent_name
                )
                & (
                    progress[
                        "replicate"
                    ].astype(int)
                    == replicate_index
                )
            )

            progress = progress.loc[
                keep_mask
            ].copy()

        progress = pd.concat(
            [
                progress,
                pd.DataFrame(
                    replicate_rows
                ),
            ],
            ignore_index=True,
        )

        progress = (
            progress
            .sort_values(
                [
                    "parent_group_name",
                    "replicate",
                    "local_cluster_code",
                ]
            )
            .reset_index(drop=True)
        )

        progress.to_csv(
            PROGRESS_FILE,
            index=False,
            compression="gzip",
        )

        completed_parent_replicates.add(key)

        elapsed_replicate = time.time() - replicate_start

        print(
            parent_name,
            "- completed partition:",
            replicate_index + 1,
            "/",
            N_RANDOM_PARTITIONS_PER_PARENT,
            "- total parent-partitions:",
            len(completed_parent_replicates),
            "/",
            total_parent_partitions,
            "- elapsed:",
            f"{elapsed_replicate:.1f}",
            "seconds",
        )

progress = pd.read_csv(PROGRESS_FILE)

completed_counts = (
    progress.groupby(
        ["parent_group_name", "replicate"]
    )[
        "local_cluster_code"
    ]
    .nunique()
)

assert len(completed_counts) == total_parent_partitions
assert (completed_counts == N_CLUSTERS_PER_PARENT).all()

expected_progress_rows = (
    len(PRIORITY_PARENTS)
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_CLUSTERS_PER_PARENT
)

assert len(progress) == expected_progress_rows

print("\nMatched-random carrier-cluster benchmark complete.")
print("Parent-partitions:", total_parent_partitions)
print("Saved rows:", len(progress))
print(
    "Current-session elapsed:",
    f"{(time.time() - benchmark_start) / 60.0:.1f}",
    "minutes",
)

print("\nCell 22.6 complete.")
print(
    "Transition: Cell 22.7 will compare each observed carrier-pattern "
    "cluster drop with its matched-random distribution."
)


In [ ]:
#@title Cell 22.7 - Compare observed carrier-pattern cluster drops with matched-random distributions
# Purpose:
# Compare each observed carrier-pattern cluster removal with matched random
# removals drawn from the same parent group.
#
# Empirical p:
#   (1 + random drops >= observed drop) / (1 + 250)
#
# BH correction is applied separately across the 10 clusters within each
# already selected parent group.

progress = pd.read_csv(PROGRESS_FILE)
observed_results = pd.read_csv(OBSERVED_RESULTS)

expected_progress_rows = (
    len(PRIORITY_PARENTS)
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_CLUSTERS_PER_PARENT
)

assert len(progress) == expected_progress_rows
assert len(observed_results) == EXPECTED_CLUSTERS

def benjamini_hochberg(p_values):
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
            dtype=float,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    result = np.empty(
        n,
        dtype=float,
    )

    result[order] = adjusted
    return result

result_rows = []

for _, observed_row in observed_results.iterrows():
    parent_name = str(
        observed_row["parent_group_name"]
    )

    local_code = int(
        observed_row["local_cluster_code"]
    )

    cluster_name = str(
        observed_row["cluster_name"]
    )

    observed_drop = float(
        observed_row["absolute_drop_after_removal"]
    )

    random_drops = (
        progress.loc[
            (
                progress[
                    "parent_group_name"
                ].astype(str)
                == parent_name
            )
            & (
                progress[
                    "local_cluster_code"
                ].astype(int)
                == local_code
            ),
            "random_drop_from_full",
        ]
        .to_numpy(dtype=float)
    )

    assert len(random_drops) == N_RANDOM_PARTITIONS_PER_PARENT

    n_equal_or_greater = int(
        np.sum(
            random_drops >= observed_drop
        )
    )

    empirical_p = float(
        (
            1
            + n_equal_or_greater
        )
        / (
            N_RANDOM_PARTITIONS_PER_PARENT
            + 1
        )
    )

    percentile = float(
        100.0
        * np.mean(
            random_drops <= observed_drop
        )
    )

    result_rows.append(
        {
            "global_cluster_code": int(
                observed_row["global_cluster_code"]
            ),
            "parent_group_name": parent_name,
            "local_cluster_code": local_code,
            "cluster_name": cluster_name,
            "n_unitigs": int(
                observed_row["n_unitigs"]
            ),
            "n_unique_carrier_patterns": int(
                observed_row[
                    "n_unique_carrier_patterns"
                ]
            ),
            "fraction_of_parent_kernel_denominator": float(
                observed_row[
                    "fraction_of_parent_kernel_denominator"
                ]
            ),
            "observed_leave_one_cluster_out_variance_fraction": float(
                observed_row[
                    "leave_one_cluster_out_variance_fraction"
                ]
            ),
            "observed_drop_after_removal": observed_drop,
            "matched_random_mean_drop": float(
                np.mean(random_drops)
            ),
            "matched_random_median_drop": float(
                np.median(random_drops)
            ),
            "matched_random_95th_percentile": float(
                np.quantile(random_drops, 0.95)
            ),
            "matched_random_99th_percentile": float(
                np.quantile(random_drops, 0.99)
            ),
            "observed_drop_percentile": percentile,
            "random_drops_equal_or_greater": n_equal_or_greater,
            "empirical_p_value": empirical_p,
        }
    )

final_results = pd.DataFrame(
    result_rows
)

final_results["within_parent_BH_q_value"] = np.nan

for parent_name in PRIORITY_PARENTS:
    mask = (
        final_results[
            "parent_group_name"
        ]
        == parent_name
    )

    assert int(mask.sum()) == N_CLUSTERS_PER_PARENT

    final_results.loc[
        mask,
        "within_parent_BH_q_value",
    ] = benjamini_hochberg(
        final_results.loc[
            mask,
            "empirical_p_value",
        ].to_numpy(dtype=float)
    )

final_results[
    "larger_drop_than_matched_random_after_within_parent_BH"
] = (
    final_results[
        "within_parent_BH_q_value"
    ]
    < 0.05
)

final_results[
    "within_parent_priority_rank"
] = (
    final_results.groupby(
        "parent_group_name"
    )[
        "empirical_p_value"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

final_results = (
    final_results
    .sort_values(
        [
            "parent_group_name",
            "empirical_p_value",
            "observed_drop_after_removal",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

final_results.to_csv(
    FINAL_RESULTS,
    index=False,
)

display(
    final_results[
        [
            "parent_group_name",
            "within_parent_priority_rank",
            "cluster_name",
            "n_unitigs",
            "n_unique_carrier_patterns",
            "fraction_of_parent_kernel_denominator",
            "observed_drop_after_removal",
            "matched_random_median_drop",
            "matched_random_95th_percentile",
            "matched_random_99th_percentile",
            "observed_drop_percentile",
            "empirical_p_value",
            "within_parent_BH_q_value",
            "larger_drop_than_matched_random_after_within_parent_BH",
        ]
    ]
)

print("\nCell 22.7 complete.")
print(
    "Transition: Cell 22.8 will perform final QC and identify "
    "carrier-pattern clusters, if any, justified for sequence-level "
    "interpretation or further refinement."
)


In [ ]:
#@title Cell 22.8 - Final QC and stopping decision
# Purpose:
# Freeze the non-coordinate refinement.
#
# A carrier-pattern cluster is carried forward only if its observed removal
# caused a larger loss than matched random removals after BH correction
# within its parent.
#
# Passing clusters are sequence-pattern priorities. They are not physical
# chromosomal locations and are not causal variant combinations.

assert CLUSTER_ASSIGNMENT.exists()
assert CLUSTER_COMPONENTS.exists()
assert CLUSTER_MANIFEST.exists()
assert CLUSTER_SUMMARY.exists()
assert OBSERVED_RESULTS.exists()
assert PROGRESS_FILE.exists()
assert FINAL_RESULTS.exists()

saved_results = pd.read_csv(FINAL_RESULTS)
progress = pd.read_csv(PROGRESS_FILE)

assert len(saved_results) == EXPECTED_CLUSTERS

expected_progress_rows = (
    len(PRIORITY_PARENTS)
    * N_RANDOM_PARTITIONS_PER_PARENT
    * N_CLUSTERS_PER_PARENT
)

assert len(progress) == expected_progress_rows

assert saved_results[
    "empirical_p_value"
].between(
    0,
    1,
).all()

assert saved_results[
    "within_parent_BH_q_value"
].between(
    0,
    1,
).all()

priority_mask = saved_results[
    "larger_drop_than_matched_random_after_within_parent_BH"
].astype(bool)

priority_clusters = (
    saved_results.loc[
        priority_mask,
        [
            "parent_group_name",
            "cluster_name",
            "n_unitigs",
            "n_unique_carrier_patterns",
            "observed_drop_after_removal",
            "empirical_p_value",
            "within_parent_BH_q_value",
        ],
    ]
    .copy()
)

parent_summary_rows = []

for parent_name in PRIORITY_PARENTS:
    n_priority = int(
        (
            priority_clusters[
                "parent_group_name"
            ]
            == parent_name
        ).sum()
    )

    parent_summary_rows.append(
        {
            "parent_group_name": parent_name,
            "clusters_tested": N_CLUSTERS_PER_PARENT,
            "clusters_passing_within_parent_BH_q_lt_0_05": n_priority,
        }
    )

parent_summary = pd.DataFrame(
    parent_summary_rows
)

qc = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "variable_unitigs": EXPECTED_UNITIGS,
            "non_coordinate_parent_groups_refined": len(PRIORITY_PARENTS),
            "carrier_pattern_clusters_per_parent": N_CLUSTERS_PER_PARENT,
            "total_clusters_tested": EXPECTED_CLUSTERS,
            "random_partitions_per_parent": N_RANDOM_PARTITIONS_PER_PARENT,
            "progress_rows": len(progress),
            "exact_parent_kernel_reconstruction_pass": True,
            "matched_unitig_count_pass": True,
            "matched_presence_count_distribution_pass": True,
            "matched_kernel_denominator_pass": True,
            "clusters_passing_within_parent_BH_q_lt_0_05": len(priority_clusters),
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    FINAL_QC,
    index=False,
)

completion_payload = {
    "status": "complete",
    "non_coordinate_parent_groups_refined": PRIORITY_PARENTS,
    "carrier_pattern_clusters_per_parent": N_CLUSTERS_PER_PARENT,
    "matched_random_partitions_per_parent": N_RANDOM_PARTITIONS_PER_PARENT,
    "clusters_passing_within_parent_BH_q_lt_0_05": priority_clusters[
        "cluster_name"
    ].astype(str).tolist(),
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(
        completion_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print("Final QC: PASS")
print("\nNon-coordinate parent summary:")
display(parent_summary)

print("\nFinal carrier-pattern cluster results:")
display(
    saved_results[
        [
            "parent_group_name",
            "within_parent_priority_rank",
            "cluster_name",
            "n_unitigs",
            "n_unique_carrier_patterns",
            "observed_drop_after_removal",
            "matched_random_95th_percentile",
            "empirical_p_value",
            "within_parent_BH_q_value",
            "larger_drop_than_matched_random_after_within_parent_BH",
        ]
    ]
)

if len(priority_clusters) > 0:
    print(
        "\nStopping decision: the following carrier-pattern clusters "
        "are supported for sequence-level interpretation or further refinement:"
    )

    for _, row in priority_clusters.iterrows():
        print(
            "-",
            row["cluster_name"],
            "- unitigs:",
            int(row["n_unitigs"]),
            "- unique carrier patterns:",
            int(row["n_unique_carrier_patterns"]),
        )

    print(
        "\nThese are carrier-pattern groups, not physical chromosomal "
        "locations. They are refinement priorities only and do not establish "
        "causal variants or causal combinations."
    )

else:
    print(
        "\nStopping decision: no UNMAPPED or MULTIMAPPED carrier-pattern "
        "cluster caused a larger loss than matched random removals after "
        "within-parent BH correction."
    )

    print(
        "In that case, the non-coordinate contribution should be treated "
        "as distributed across carrier-pattern structure rather than "
        "localized to one cluster by this analysis."
    )

print(
    "\nNotebook 22 ends here. Review these results before any deeper "
    "sequence interpretation and before constructing the chromosome-MIC "
    "visualisation notebook."
)
